# RASTA CAD OCTA Classification — Phase 1 Dataset Analysis

This notebook is the runnable project entry point. Phase 1 audits the RASTA OCTA dataset and clinical metadata before any modelling.

**Important:** patient-level splitting and modelling are intentionally not executed in this phase. Set `DATA_ROOT` to the RASTA dataset folder on the GPU machine before running.

In [ ]:
from pathlib import Path
import os
import json
import re
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# -----------------------------
# Notebook-local utilities
# -----------------------------

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)


def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def _jsonable(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    return obj


def save_json(obj, path, indent=2):
    path = Path(path)
    ensure_dir(path.parent)
    path.write_text(json.dumps(_jsonable(obj), indent=indent, sort_keys=True))


# -----------------------------
# RASTA manifest scanning
# -----------------------------
LAYER_ALIASES = {
    'sup': ('sup', 'super', 'superficial', 'svc'),
    'deep': ('deep', 'dvc'),
    'cc': ('cc', 'choriocap', 'choriocapillaris'),
}
EYES = ('OD', 'OS')
IMAGE_EXTS = {'.bmp', '.png', '.jpg', '.jpeg', '.tif', '.tiff'}


def _normalise_name(name):
    return re.sub(r'[^a-z0-9]+', ' ', str(name).lower()).strip()


def infer_eye(filename):
    tokens = set(_normalise_name(filename).upper().split())
    for eye in EYES:
        if eye in tokens:
            return eye
    m = re.search(r'(?:^|[^A-Z])(OD|OS)(?:[^A-Z]|$)', str(filename).upper())
    return m.group(1) if m else None


def infer_layer(filename):
    norm = _normalise_name(filename)
    tokens = set(norm.split())
    for layer, aliases in LAYER_ALIASES.items():
        if any(alias in tokens or alias in norm for alias in aliases):
            return layer
    return None


def cohort_label_from_folder(folder_name):
    m = re.match(r'(\d+)[_\-\s]*(.*)', str(folder_name))
    if not m:
        return str(folder_name), None
    idx = int(m.group(1)) - 1
    name = m.group(2) or str(folder_name)
    return name, idx


def scan_image_manifest(root, require_all_layers=True):
    """Scan RASTA-style folders into one row per patient-eye.

    Expected structure: root/cohort_folder/patient_id/*.bmp.
    Naming is matched case-insensitively and tolerates spaces/underscores,
    e.g. `cc OD.bmp`, `Cc_OD.bmp`, `deep OS.bmp`.
    """
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f'Dataset root does not exist: {root}')

    records = []
    cohort_dirs = sorted([p for p in root.iterdir() if p.is_dir() and not p.name.startswith('.')], key=lambda p: p.name)
    label_map = {}

    for fallback_label, cohort_dir in enumerate(cohort_dirs):
        cohort_name, parsed_label = cohort_label_from_folder(cohort_dir.name)
        label = parsed_label if parsed_label is not None else fallback_label
        label_map[cohort_name] = label

        patient_dirs = sorted([p for p in cohort_dir.iterdir() if p.is_dir() and not p.name.startswith('.')], key=lambda p: p.name)
        for patient_dir in patient_dirs:
            per_eye = {eye: {'sup_path': None, 'deep_path': None, 'cc_path': None} for eye in EYES}
            for file in patient_dir.iterdir():
                if not file.is_file() or file.name.startswith('._') or file.suffix.lower() not in IMAGE_EXTS:
                    continue
                eye = infer_eye(file.name)
                layer = infer_layer(file.name)
                if eye is None or layer is None:
                    continue
                per_eye[eye][f'{layer}_path'] = str(file)

            for eye, paths in per_eye.items():
                has_any = any(paths.values())
                has_all = all(paths.values())
                if not has_any:
                    continue
                if require_all_layers and not has_all:
                    continue
                records.append({
                    'patient_id': patient_dir.name,
                    'cohort': cohort_name,
                    'cohort_folder': cohort_dir.name,
                    'label': label,
                    'eye': eye,
                    **paths,
                    'has_all_layers': has_all,
                })

    df = pd.DataFrame(records)
    if not df.empty:
        df = df.sort_values(['label', 'patient_id', 'eye']).reset_index(drop=True)
    df.attrs['label_map'] = label_map
    return df


# -----------------------------
# Clinical metadata helpers
# -----------------------------

def load_clinical_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() in {'.xlsx', '.xls'}:
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)
    df = df.rename(columns={c: str(c).strip() for c in df.columns})
    if 'ID' in df.columns and 'patient_id' not in df.columns:
        df = df.rename(columns={'ID': 'patient_id'})
    return df


def attach_clinical(manifest, clinical):
    if clinical is None or clinical.empty:
        return manifest.copy()
    if 'patient_id' not in clinical.columns:
        raise ValueError('Clinical table must contain ID or patient_id column')
    return manifest.merge(clinical, on='patient_id', how='left', validate='many_to_one')


def default_clinical_columns(df):
    preferred = [
        'Age', 'Sex', 'Congestive heart failure', 'Hypertension',
        'Diabetes mellitus', 'Stroke', 'Vascular disease', 'Body mass index',
        'CHA2DS2-VASc', 'Obstructive sleep apnea syndrom', 'Smoking', 'Dyslipidemia',
    ]
    stripped = {str(c).strip(): c for c in df.columns}
    cols = [stripped[c] for c in preferred if c in stripped]
    if cols:
        return cols
    exclude = {
        'patient_id', 'cohort', 'cohort_folder', 'label', 'eye',
        'sup_path', 'deep_path', 'cc_path', 'fold', 'has_all_layers'
    }
    return [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]


def make_age_decade(age):
    age = pd.to_numeric(age, errors='coerce')
    decade = (age.fillna(-1) // 10).astype(int) * 10
    return decade.astype(str).where(age.notna(), 'age_missing')


def dataset_audit(df, clinical_cols=None):
    audit = {}
    audit['n_eye_samples'] = int(len(df))
    audit['n_patients'] = int(df['patient_id'].nunique()) if 'patient_id' in df else 0

    if 'cohort' in df:
        audit['class_counts_eye'] = df['cohort'].value_counts().to_dict()
        patient_counts = df.drop_duplicates('patient_id')['cohort'].value_counts().to_dict()
        audit['class_counts_patient'] = patient_counts
        counts = np.array(list(patient_counts.values()), dtype=float)
        audit['class_imbalance_ratio_patient'] = float(counts.max() / max(counts.min(), 1)) if len(counts) else np.nan

    if {'patient_id', 'eye'}.issubset(df.columns):
        eyes_per_patient = df.groupby('patient_id')['eye'].agg(lambda x: '+'.join(sorted(set(x))))
        audit['bilateral_availability'] = eyes_per_patient.value_counts().to_dict()

    layer_cols = [c for c in ['sup_path', 'deep_path', 'cc_path'] if c in df]
    audit['layer_missing_counts'] = {c: int(df[c].isna().sum()) for c in layer_cols}

    if clinical_cols:
        missing = df[list(clinical_cols)].isna().sum().sort_values(ascending=False)
        audit['clinical_missing_counts'] = missing.astype(int).to_dict()
        if 'cohort' in df:
            by_cohort = {}
            for cohort, group in df.groupby('cohort'):
                by_cohort[cohort] = group[list(clinical_cols)].isna().sum().astype(int).to_dict()
            audit['clinical_missing_by_cohort'] = by_cohort

    if 'patient_id' in df and 'cohort' in df:
        leakage = df.groupby('patient_id')['cohort'].nunique()
        audit['patients_in_multiple_cohorts'] = leakage[leakage > 1].index.tolist()

    return audit


# -----------------------------
# Patient-level fold helpers
# -----------------------------

def make_stratification_key(df, age_col='Age', sex_col='Sex'):
    label = df['label'].astype(str)
    key = label.copy()
    if age_col and age_col in df.columns:
        key = key + '_' + make_age_decade(df[age_col])
    if sex_col and sex_col in df.columns:
        sex = df[sex_col].astype('object').where(df[sex_col].notna(), 'sex_missing').astype(str)
        key = key + '_' + sex
    counts = key.value_counts()
    rare = key.map(counts) < 2
    return key.where(~rare, label)


def assign_patient_folds(df, n_splits=5, seed=42, age_col='Age', sex_col='Sex'):
    if not {'patient_id', 'label'}.issubset(df.columns):
        raise ValueError('df must include patient_id and label columns')

    out = df.copy().reset_index(drop=True)
    patient_df = out.drop_duplicates('patient_id').reset_index(drop=True)
    y = make_stratification_key(patient_df, age_col=age_col, sex_col=sex_col)
    groups = patient_df['patient_id'].astype(str)
    folds = np.full(len(patient_df), -1, dtype=int)

    try:
        from sklearn.model_selection import StratifiedGroupKFold
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for fold, (_, val_idx) in enumerate(splitter.split(patient_df, y, groups)):
            folds[val_idx] = fold
    except Exception:
        # Deterministic label/stratum-balanced fallback at patient level.
        rng = np.random.default_rng(seed)
        fold_sizes = np.zeros(n_splits, dtype=int)
        for _, grp in patient_df.assign(_key=y.values).groupby('_key', sort=False):
            idx = grp.index.to_numpy()
            rng.shuffle(idx)
            for i in idx:
                f = int(np.argmin(fold_sizes))
                folds[i] = f
                fold_sizes[f] += 1

    patient_to_fold = dict(zip(patient_df['patient_id'], folds.astype(int)))
    out['fold'] = out['patient_id'].map(patient_to_fold).astype(int)
    validate_patient_folds(out, n_splits=n_splits)
    return out


def validate_patient_folds(df, n_splits=None):
    per_patient = df.groupby('patient_id')['fold'].nunique()
    bad = per_patient[per_patient > 1]
    if len(bad):
        raise AssertionError(f'Patient leakage across folds: {bad.index.tolist()[:10]}')
    if n_splits is not None:
        got = set(df['fold'].dropna().astype(int).unique())
        expected = set(range(n_splits))
        if not got.issubset(expected):
            raise AssertionError(f'Unexpected fold ids: {got - expected}')


seed_everything(42)
sns.set_theme(style='whitegrid', context='notebook')
PROJECT_ROOT = Path.cwd()

## Configuration — edit paths here

This notebook will be run on Windows, so the dataset path is intentionally kept in one easy-to-edit cell below.

Use a raw string for Windows paths, for example:

```python
DATA_ROOT_STR = r"D:\RASTA\Rasta dataset"
```

If `DATA_ROOT_STR` is left empty, the notebook will try the environment variable `RASTA_DATA_ROOT`, then the relative folder `Rasta dataset`.


In [ ]:
# ============================================================
# EDIT THESE PATHS ON THE WINDOWS MACHINE
# ============================================================

# 1) RASTA image dataset folder. Example Windows path:
# DATA_ROOT_STR = r"D:\RASTA\Rasta dataset"
DATA_ROOT_STR = r""

# 2) Clinical spreadsheet path. If it is in the same folder as this notebook,
#    the default below should work. Example Windows path:
# CLINICAL_PATH_STR = r"D:\RASTA\table_for_publication .xlsx"
CLINICAL_PATH_STR = r"table_for_publication .xlsx"

# 3) Output folder for Phase 1 reports/plots. Relative paths are saved beside the notebook.
OUTPUT_DIR_STR = r"outputs/phase1_dataset_analysis"

# Optional runtime settings
REQUIRE_ALL_LAYERS_FOR_MODEL = True
QC_MAX_IMAGES_PER_LAYER = None  # set e.g. 500 for a quicker smoke-test audit


def choose_path(user_value, env_name, default_value):
    """Choose path in order: explicit notebook value -> environment variable -> default."""
    value = str(user_value).strip() if user_value is not None else ""
    if not value:
        value = os.environ.get(env_name, default_value)
    return Path(value).expanduser()


DATA_ROOT = choose_path(DATA_ROOT_STR, 'RASTA_DATA_ROOT', 'Rasta dataset')
CLINICAL_PATH = choose_path(CLINICAL_PATH_STR, 'RASTA_CLINICAL_PATH', 'table_for_publication .xlsx')
OUTPUT_DIR = ensure_dir(Path(OUTPUT_DIR_STR).expanduser())

print('DATA_ROOT      =', DATA_ROOT.resolve() if DATA_ROOT.exists() else DATA_ROOT)
print('CLINICAL_PATH  =', CLINICAL_PATH.resolve() if CLINICAL_PATH.exists() else CLINICAL_PATH)
print('OUTPUT_DIR     =', OUTPUT_DIR.resolve())
print()
print('To change paths on Windows, edit DATA_ROOT_STR and CLINICAL_PATH_STR in this cell only.')


## Load clinical metadata

The provided publication spreadsheet contains demographics, CAD risk factors, and precomputed OCTA biomarkers. Missingness is preserved for later learnable mask-token handling.

In [ ]:
clinical_df = load_clinical_table(CLINICAL_PATH)
print('Clinical shape:', clinical_df.shape)
display(clinical_df.head())
print('Columns:')
print(list(clinical_df.columns))

## Scan OCTA files and create patient-eye manifest

Each row is one eye. OD/OS are represented separately but retain the same `patient_id` for leakage-safe fold assignment later. The scanner tolerates RASTA naming variants such as `cc OD.bmp`, `Cc_OD.bmp`, and case differences.

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f'DATA_ROOT does not exist: {DATA_ROOT}\n\n'
        'Edit DATA_ROOT_STR in the Configuration cell above. '
        'For Windows, use a raw string like: r"D:\\RASTA\\Rasta dataset"'
    )

manifest_all = scan_image_manifest(DATA_ROOT, require_all_layers=False)
manifest = scan_image_manifest(DATA_ROOT, require_all_layers=REQUIRE_ALL_LAYERS_FOR_MODEL)
manifest_clin = attach_clinical(manifest, clinical_df)

manifest_all.to_csv(OUTPUT_DIR / 'manifest_all_detected_eyes.csv', index=False)
manifest_clin.to_csv(OUTPUT_DIR / 'manifest_model_ready_with_clinical.csv', index=False)

print('All detected eye rows:', manifest_all.shape)
print('Model-ready rows with all layers:', manifest.shape)
display(manifest_clin.head())


## Core dataset audit

This covers total samples, cohort imbalance, bilateral availability, layer availability, clinical missingness, and possible patient leakage across cohorts.

In [ ]:
clinical_cols = default_clinical_columns(manifest_clin)
audit = dataset_audit(manifest_clin, clinical_cols=clinical_cols)
save_json(audit, OUTPUT_DIR / 'dataset_audit.json')

print(json.dumps(audit, indent=2, default=str))

class_eye = manifest_clin['cohort'].value_counts().rename_axis('cohort').reset_index(name='eye_count')
class_patient = manifest_clin.drop_duplicates('patient_id')['cohort'].value_counts().rename_axis('cohort').reset_index(name='patient_count')
class_table = class_patient.merge(class_eye, on='cohort', how='outer')
class_table.to_csv(OUTPUT_DIR / 'class_distribution.csv', index=False)
display(class_table)

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=class_table, x='cohort', y='patient_count', color='#4C72B0')
plt.xticks(rotation=30, ha='right')
plt.title('Patient count per cohort')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_distribution_patients.png', dpi=200)
plt.show()

eyes_per_patient = manifest_clin.groupby('patient_id')['eye'].agg(lambda s: '+'.join(sorted(set(s))))
bilat = eyes_per_patient.value_counts().rename_axis('eye_availability').reset_index(name='patient_count')
bilat.to_csv(OUTPUT_DIR / 'bilateral_availability.csv', index=False)
display(bilat)

## Clinical missingness and demographic distributions

In [ ]:
missing_overall = manifest_clin[clinical_cols].isna().mean().sort_values(ascending=False).rename('missing_fraction')
missing_by_cohort = manifest_clin.groupby('cohort')[clinical_cols].apply(lambda g: g.isna().mean()).reset_index()
missing_overall.to_csv(OUTPUT_DIR / 'clinical_missing_overall.csv')
missing_by_cohort.to_csv(OUTPUT_DIR / 'clinical_missing_by_cohort.csv', index=False)
display(missing_overall.to_frame())
display(missing_by_cohort)

plt.figure(figsize=(12, max(4, 0.025 * len(manifest_clin))))
sns.heatmap(manifest_clin[clinical_cols].isna(), cbar=False, yticklabels=False)
plt.title('Missing clinical data heatmap (eye rows × features)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'clinical_missingness_heatmap.png', dpi=200)
plt.show()

In [ ]:
demo_cols = [c for c in ['Age', 'Sex', 'Body mass index'] if c in manifest_clin.columns]
demo_summary = manifest_clin.drop_duplicates('patient_id').groupby('cohort')[demo_cols].agg(['count', 'mean', 'std', 'median', 'min', 'max'])
demo_summary.to_csv(OUTPUT_DIR / 'demographic_summary_by_cohort.csv')
display(demo_summary)

if 'Age' in manifest_clin.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=manifest_clin.drop_duplicates('patient_id'), x='cohort', y='Age')
    plt.xticks(rotation=30, ha='right')
    plt.title('Age distribution per cohort')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'age_distribution_by_cohort.png', dpi=200)
    plt.show()

if 'Body mass index' in manifest_clin.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=manifest_clin.drop_duplicates('patient_id'), x='cohort', y='Body mass index')
    plt.xticks(rotation=30, ha='right')
    plt.title('BMI distribution per cohort')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'bmi_distribution_by_cohort.png', dpi=200)
    plt.show()

## Image resolution and lightweight quality audit

This reads OCTA images to summarize resolution, intensity range, standard deviation, and zero-variance/unreadable counts. It does **not** train or run any model.

In [ ]:
def audit_image_paths(df, max_per_layer=None):
    rows = []
    for layer, col in [('sup', 'sup_path'), ('deep', 'deep_path'), ('cc', 'cc_path')]:
        paths = df[col].dropna().tolist()
        if max_per_layer is not None:
            paths = paths[:max_per_layer]
        for p in paths:
            rec = {'layer': layer, 'path': p}
            try:
                with Image.open(p) as img:
                    arr = np.asarray(img.convert('L'), dtype=np.float32)
                rec.update({
                    'height': int(arr.shape[0]), 'width': int(arr.shape[1]),
                    'mean': float(arr.mean()), 'std': float(arr.std()),
                    'min': float(arr.min()), 'max': float(arr.max()),
                    'zero_variance': bool(arr.std() <= 1e-6),
                    'readable': True,
                })
            except Exception as e:
                rec.update({'readable': False, 'error': repr(e)})
            rows.append(rec)
    return pd.DataFrame(rows)

image_qc = audit_image_paths(manifest_clin, max_per_layer=QC_MAX_IMAGES_PER_LAYER)
image_qc.to_csv(OUTPUT_DIR / 'image_quality_audit.csv', index=False)
display(image_qc.head())

qc_summary = image_qc.groupby('layer').agg(
    n=('path', 'count'),
    readable=('readable', 'sum'),
    zero_variance=('zero_variance', 'sum'),
    median_height=('height', 'median'),
    median_width=('width', 'median'),
    median_std=('std', 'median'),
).reset_index()
qc_summary.to_csv(OUTPUT_DIR / 'image_quality_summary.csv', index=False)
display(qc_summary)

if not image_qc.empty and 'std' in image_qc:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=image_qc[image_qc['readable']], x='layer', y='std')
    plt.title('Image intensity standard deviation by OCTA layer')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'image_std_by_layer.png', dpi=200)
    plt.show()

## Biomarker distributions

The clinical spreadsheet includes OCTA biomarker columns such as density, perfusion, and FAZ metrics. These are summarized and plotted by cohort as available.

In [ ]:
biomarker_pattern = re.compile(r'^(FAZ|Faz|DENS_|PERF_)')
biomarker_cols = [c for c in manifest_clin.columns if biomarker_pattern.search(str(c))]
print('Detected biomarker columns:', len(biomarker_cols))
print(biomarker_cols[:20])

if biomarker_cols:
    biomarker_summary = manifest_clin.groupby('cohort')[biomarker_cols].agg(['count', 'mean', 'std', 'median'])
    biomarker_summary.to_csv(OUTPUT_DIR / 'biomarker_summary_by_cohort.csv')
    display(biomarker_summary.iloc[:, :min(16, biomarker_summary.shape[1])])

    plot_cols = biomarker_cols[:12]
    for col in plot_cols:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=manifest_clin, x='cohort', y=col)
        plt.xticks(rotation=30, ha='right')
        plt.title(f'{col} by cohort')
        plt.tight_layout()
        safe = re.sub(r'[^A-Za-z0-9_.-]+', '_', col)
        plt.savefig(OUTPUT_DIR / f'biomarker_boxplot_{safe}.png', dpi=200)
        plt.show()

## Confounder and leakage checks

We inspect age/sex imbalance and verify that no patient ID appears in multiple cohorts. A preliminary patient-level stratified fold assignment is generated for review only.

In [ ]:
patient_level = manifest_clin.drop_duplicates('patient_id').copy()
leakage = patient_level.groupby('patient_id')['cohort'].nunique()
leaked_patients = leakage[leakage > 1].index.tolist()
print('Patients appearing in multiple cohorts:', len(leaked_patients))
print(leaked_patients[:20])

confounder_tables = {}
if 'Age' in patient_level.columns:
    confounder_tables['age_by_cohort'] = patient_level.groupby('cohort')['Age'].describe()
    display(confounder_tables['age_by_cohort'])
if 'Sex' in patient_level.columns:
    confounder_tables['sex_by_cohort'] = pd.crosstab(patient_level['cohort'], patient_level['Sex'], normalize='index')
    display(confounder_tables['sex_by_cohort'])

for name, table in confounder_tables.items():
    table.to_csv(OUTPUT_DIR / f'{name}.csv')

fold_df = assign_patient_folds(manifest_clin, n_splits=5, seed=42, age_col='Age' if 'Age' in manifest_clin else None, sex_col='Sex' if 'Sex' in manifest_clin else None)
validate_patient_folds(fold_df, n_splits=5)
fold_df[['patient_id', 'eye', 'cohort', 'label', 'fold']].to_csv(OUTPUT_DIR / 'preliminary_patient_level_folds.csv', index=False)
display(pd.crosstab(fold_df['fold'], fold_df['cohort']))
print('Zero patient overlap across folds verified.')

## Phase 1 completion checklist

After running this notebook section on the dataset machine, review the files in `outputs/phase1_dataset_analysis/` before proceeding to Phase 2 preprocessing and dataset object finalization.